In [1]:
import sys
import numpy as np
import pygame
import gymnasium as gym

from gymnasium.wrappers import RecordVideo


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
import sys
import numpy as np

if "./gym_env" not in sys.path:
    sys.path.insert(0, "./gym_env")

from gym_env.envs.grid_world import GridWorldEnv, Actions

def get_step_outcome(env, state, action):
    """Computes (next_state, reward, terminated) for a given (x, y) state and action."""
    pos = np.array(state, dtype=int)
    # Terminal states (target or bad cells) have 0 future transitions
    if (
        np.array_equal(pos, env._target_location)
        or np.array_equal(pos, env._bad1_location)
        or np.array_equal(pos, env._bad2_location)
    ):
        return tuple(pos), 0.0, True

    direction = env._action_to_direction[action]
    next_pos = np.clip(pos + direction, 0, env.size - 1)

    if np.array_equal(next_pos, env._target_location):
        return tuple(next_pos), 1.0, True
    elif np.array_equal(next_pos, env._bad1_location) or np.array_equal(next_pos, env._bad2_location):
        return tuple(next_pos), -1.0, True
    return tuple(next_pos), 0.0, False


def policy_iteration(env, gamma=0.9, theta=1e-6):
    policy = np.zeros((env.size, env.size), dtype=int)
    V = np.zeros((env.size, env.size), dtype=float)
    iterations = 0

    while True:
        iterations += 1
        # 1. Policy Evaluation
        while True:
            delta = 0.0
            for x in range(env.size):
                for y in range(env.size):
                    v_old = V[x, y]
                    (nx, ny), reward, terminated = get_step_outcome(env, (x, y), policy[x, y])
                    V[x, y] = reward + (0.0 if terminated else gamma * V[nx, ny])
                    delta = max(delta, abs(v_old - V[x, y]))
            if delta < theta:
                break

        # 2. Policy Improvement
        policy_stable = True
        for x in range(env.size):
            for y in range(env.size):
                old_action = policy[x, y]
                q_values = np.zeros(env.action_space.n)
                for a in range(env.action_space.n):
                    (nx, ny), reward, terminated = get_step_outcome(env, (x, y), a)
                    q_values[a] = reward + (0.0 if terminated else gamma * V[nx, ny])

                best_action = int(np.argmax(q_values))
                policy[x, y] = best_action
                if best_action != old_action and not np.isclose(q_values[best_action], q_values[old_action]):
                    policy_stable = False

        if policy_stable:
            break

    return policy, V, iterations

In [ ]:
from gymnasium.wrappers import RecordVideo

env = GridWorldEnv(render_mode="rgb_array", size=5)
video_env = RecordVideo(
    env,
    video_folder="./videos",
    name_prefix="grid_maze_agent",
    episode_trigger=lambda ep: True,
)

obs, info = video_env.reset(seed=42)
policy, V, iters = policy_iteration(video_env.unwrapped, gamma=0.9)
print(f"Policy Iteration converged in {iters} iterations.")

terminated = False
steps = 0
while not terminated and steps < 50:
    agent_x, agent_y = obs["agent"]
    action = int(policy[agent_x, agent_y])
    obs, reward, terminated, truncated, info = video_env.step(action)
    steps += 1

video_env.close()
print(f"Episode finished in {steps} steps with final reward {reward}. Video saved to ./videos/")

Policy Iteration converged in 4 iterations.
Episode finished in 4 steps with final reward 1. Video saved to ./videos/


In [4]:
from gymnasium.wrappers import RecordVideo

video_env = GridWorldEnv(render_mode="rgb_array", size=5)
video_env = RecordVideo(
    video_env,
    video_folder="./videos",
    name_prefix="grid_maze_policy_iteration",
    episode_trigger=lambda episode_id: True,
)

num_episodes = 5
for ep in range(num_episodes):
    obs, info = video_env.reset(seed=100 + ep)
    
    # Unpack 3 values matching policy_iteration()
    policy, V, iters = policy_iteration(video_env.unwrapped, gamma=0.9)

    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0

    while not (terminated or truncated):
        agent_x, agent_y = obs["agent"]
        action = int(policy[agent_x, agent_y])
        obs, reward, terminated, truncated, info = video_env.step(action)
        total_reward += reward
        steps += 1

    print(f"Episode {ep + 1}: Converged in {iters} iterations | Steps: {steps} | Total Reward = {total_reward}")

video_env.close()
print("Videos successfully saved to ./videos/")

Episode 1: Converged in 7 iterations | Steps: 5 | Total Reward = 1.0


c:\Users\gamin\OneDrive\Desktop\gam3a\fall 2026\rl\labs\lab 2\RL-lab2\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:298: UserWarning: WARN: Overwriting existing videos at c:\Users\gamin\OneDrive\Desktop\gam3a\fall 2026\rl\labs\lab 2\RL-lab2\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode 2: Converged in 5 iterations | Steps: 5 | Total Reward = 1.0
Episode 3: Converged in 5 iterations | Steps: 2 | Total Reward = 1.0
Episode 4: Converged in 8 iterations | Steps: 2 | Total Reward = 1.0
Episode 5: Converged in 7 iterations | Steps: 3 | Total Reward = 1.0
Videos successfully saved to ./videos/



## Questions

### Q1: What is the state-space size of the $5 \times 5$ Grid Maze problem?
* **Fixed Maze Instance (Per Episode):** $25\text{ states}$ ($5 \times 5$ -3 grid positions for the agent).
* **Full Joint State Space (All Initializations):**
  * **Raw observation space:** $5^2 \times 5^2 \times 5^2 \times 5^2 = 25^4 = \mathbf{390,625\text{ states}}$.
  * **Valid unique states (no position overlap):** $25 \times 24 \times 23 \times 22 = \mathbf{303,600\text{ states}}$ (or $\mathbf{151,800\text{ states}}$ if the two bad cells are treated as identical).

---

### Q2: How to optimize policy iteration for the Grid Maze problem?
* **In-Place (Gauss-Seidel) Updates:** Update value function values $V(s)$ in-place during evaluation sweeps to accelerate convergence.
* **Truncated Policy Evaluation:** Stop policy evaluation early after a fixed number of sweeps $k$ or a larger threshold $\theta$ rather than waiting for full mathematical convergence.
* **Terminal State Skipping:** Exclude terminal states (Target/Bad cells) from value update sweeps by keeping $V(\text{terminal}) = 0$.

---

### Q3: How many iterations did it take to converge on a stable policy for $5 \times 5$ maze?
* **Single Run (Seed 42):** Converged in **$4$ iterations**.
* **5-Episode Run (Seeds 100–104):** Converged in **$5$ to $8$ iterations** (average: **$6.4$ iterations**).

| Episode | Seed | Convergence Iterations | Step Count | Total Reward |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 100 | 7 | 5 | +1.0 |
| 2 | 101 | 5 | 5 | +1.0 |
| 3 | 102 | 5 | 2 | +1.0 |
| 4 | 103 | 8 | 2 | +1.0 |
| 5 | 104 | 7 | 3 | +1.0 |

---

### Q4: Explain, with an example, how policy iteration behaves with multiple goal cells.

Policy Iteration evaluates paths to all goal cells simultaneously by propagating expected returns backward so we have two outcomes if the goals are of equal value or unequal value.

* **Equal Goal Rewards (Goal A = Goal B = $+1.0$):**
  * **Attraction Basins (Voronoi Regions):** The grid partitions based purely on shortest path distance because return decays by step count ($\gamma^d \cdot 1.0$). States automatically route toward the nearest goal.

* **Unequal Goal Rewards (Reward vs. Distance Trade-off):**
  * **Example:** Goal A (dist $d=1$) gives reward $+1.0$; Goal B (dist $d=4$) gives reward $+10.0$ ($\gamma = 0.9$).
  * $\text{Return to Goal A} = 1.0$
  * $\text{Return to Goal B} = 0.9^3 \times 10.0 = \mathbf{7.29}$
  * **Behavior:** The agent will bypass Goal A right next to it to walk 4 steps to collect Goal B because $7.29 > 1.0$.

---

### Q5: Can policy iteration work on a $100 \times 100$ maze? Explain why?
* **Fixed Grid Instance:** **Yes.** A single grid layout has $100 \times 100 = 10,000$ states, which Policy Iteration solves easily in seconds.
* **Full Joint State Space:** **No.** Randomizing target and bad cell positions yields $10,000 \times 9,999 \times 9,998 \times 9,997 \approx \mathbf{10^{16}\text{ states}}$, causing Bellman's **Curse of Dimensionality** (exceeding system memory and compute limits).

---

### Q6: Can policy iteration work on a continuous-space maze? Explain why?
* **Direct Tabular Policy Iteration:** **No.** Continuous spaces contain an uncountably infinite number of states $(x, y) \in \mathbb{R}^2$, rendering finite lookup tables for $V(s)$ and $\pi(s)$ impossible.
* **Adaptations:** Requires spatial discretization (e.g., tile coding) or **Approximate Policy Iteration**.

---

### Q7: Can policy iteration work with moving bad cells (like Pacman moving ghosts)? Explain why?
* **Standard Spatial State $(x_a, y_a)$:** **No.** Omitting ghost positions breaks the **Markov Property**, causing environmental transition dynamics to change unpredictably over time.
* **Augmented Joint State Space:** **Yes (In Theory).** If states include ghost positions $s = (x_a, y_a, x_{g1}, y_{g1}, x_{g2}, y_{g2})$ and ghost transition probabilities are fully known.
* **Limitation:** State space explodes dramatically ($25^3 = 15,625$ states for a $5 \times 5$ grid; $10^{12}$ states for $100 \times 100$). Stochastic or hostile ghost behaviors require Game Theory formulations (e.g., Minimax-Q).

---

## 2. Deliverables Checklist & Gap Analysis

| Deliverable | Status | Action Required |
| :--- | :--- | :--- |
| **Python Code** (`GridWorldEnv`, `policy_iteration`) | **Complete** | Included in `grid_world.py` and `Lab2.ipynb`. |
| **Recorded Videos** | **Partial** | MP4 files are output to `./videos/`. Ensure they are saved clean without open handles. |
| **GitHub Repository** | **Missing** | Create a repository containing `grid_world.py`, `Lab2.ipynb`, and `requirements.txt`. |
| **Report Document** | **Ready** | Export/submit this Markdown document as a PDF/Word report. |